In [1]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 31.5 MB/s eta 0:00:00:00:0100:01


In [2]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers import BitsAndBytesConfig
import torch
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,               # reduce precision
    bnb_4bit_compute_dtype=torch.float16,   # use float16 where needed
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)



2026-01-30 20:53:52.417059: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769806432.772960      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769806432.876828      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769806433.720286      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769806433.720329      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769806433.720333      55 computation_placer.cc:177] computation placer alr

In [3]:
model_id = "Qwen/Qwen2-VL-7B-Instruct"


In [4]:
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quant_config
)


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

## Dataset Indiana PRO reports 

In [5]:
import pandas as pd
import numpy as np

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import AutoProcessor


In [7]:
images_folder = "/kaggle/input/chest-xrays-indiana-university/images/images_normalized"  # adjust path
csv_path = "/kaggle/input/chest-xrays-indiana-university/indiana_reports.csv"

In [8]:
df = pd.read_csv(csv_path)

df.head()
df=df[['uid','findings','impression']]
df.head()
dft=pd.read_csv("/kaggle/input/indiana-pro-reports/openi_captions.txt")
df['image']=pd.read_csv("/kaggle/input/indiana-pro-reports/openi_captions.txt")['image']
df.head()
print(pd.read_csv("/kaggle/input/indiana-pro-reports/openi_captions.txt")['caption'][0])
print(df['findings'][0])

the cardiac silhouette and mediastinum size are within normal limits. there is no pulmonary edema. there is no focal consolidation. there are no xxxx of a pleural effusion. there is no evidence of pneumothorax.
The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no XXXX of a pleural effusion. There is no evidence of pneumothorax.


In [9]:
df.head()

,uid,findings,impression,image
0,1,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.,1_IM-0001-4001.dcm.png
1,2,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.,2_IM-0652-1001.dcm.png
2,3,NaN,"No displaced rib fractures, pneumothorax, or p...",4_IM-2050-1001.dcm.png
3,4,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...,5_IM-2117-1003002.dcm.png
4,5,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.,6_IM-2192-1001.dcm.png


In [10]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)


In [11]:
print(train_df['image'].isna().sum())
import os
train_df = train_df[train_df['image'].apply(lambda x: os.path.exists(f"{images_folder}/{x}"))]
train_df = train_df.reset_index(drop=True)
print(train_df['image'].isna().sum())
print(len(train_df),len(test_df))

430
0
2650 771


In [12]:

# train/test split (80/20)

# -----------------------------
# 3️⃣ Dataset
# -----------------------------
#train_dataset = ChestXrayDataset(train_df, images_folder, processor)

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, images_folder, processor):
        self.data = dataframe.reset_index(drop=True)
        self.images_folder = images_folder
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = f"{self.images_folder}/{row['image']}"
        image = Image.open(img_path).convert("RGB")
    
        report = str(row.get('findings', "")) + " " + str(row.get('impression', ""))
    
        prompt = "<|vision_start|><|image_pad|><|vision_end|> Generate a radiology report for this chest X-ray."
        text = prompt + " " + report
    
        encoding = self.processor(
    images=image,
            text=text,
            padding=False,         # keep variable lengths
            return_tensors="pt"
)
    
        item = {k: v.squeeze(0) for k, v in encoding.items()}
    
        labels = item["input_ids"].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        item["labels"] = labels
    
        return item
    



batch_size = 2  # start small on Kaggle
train_dataset = ChestXrayDataset(train_df, images_folder, processor)
test_dataset = ChestXrayDataset(test_df, images_folder, processor)



In [31]:
def collate_fn(batch):
    # Find max lengths
    max_input_length = max(len(item['input_ids']) for item in batch)
    max_label_length = max(len(item['labels']) for item in batch)
    
    # Pad input_ids and attention_mask
    padded_input_ids = []
    padded_attention_mask = []
    for item in batch:
        input_ids = item['input_ids']
        attention_mask = item['attention_mask']
        
        padding_length = max_input_length - len(input_ids)
        
        padded_input_ids.append(
            torch.cat([input_ids, torch.full((padding_length,), processor.tokenizer.pad_token_id)])
        )
        
        padded_attention_mask.append(
            torch.cat([attention_mask, torch.zeros(padding_length, dtype=attention_mask.dtype)])
        )
    
    # Pad labels
    padded_labels = []
    for item in batch:
        labels = item['labels']
        padding_length = max_label_length - len(labels)
        padded_labels.append(
            torch.cat([labels, torch.full((padding_length,), -100)])
        )
    
    # Pad pixel_values (NEW - this is what was missing)
    pixel_values_list = [item['pixel_values'] for item in batch]
    max_pixel_length = max(pv.shape[0] for pv in pixel_values_list)
    
    padded_pixel_values = []
    for pv in pixel_values_list:
        if pv.shape[0] < max_pixel_length:
            padding_size = max_pixel_length - pv.shape[0]
            padding = torch.zeros((padding_size, *pv.shape[1:]), dtype=pv.dtype)
            padded_pv = torch.cat([pv, padding], dim=0)
        else:
            padded_pv = pv
        padded_pixel_values.append(padded_pv)
    
    return {
        'input_ids': torch.stack(padded_input_ids),
        'attention_mask': torch.stack(padded_attention_mask),
        'labels': torch.stack(padded_labels),
        'pixel_values': torch.stack(padded_pixel_values)
    }

# Keep your DataLoader with this custom collate_fn
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [ ]:
train_loader = DataLoader(
    train_dataset, 
    batch_size=2, 
    shuffle=True,
    collate_fn=collate_fn  # Add this line
)



In [32]:
for i in range(5):
    s = train_dataset[i]
    print(f"Sample {i} | image shape: {s['pixel_values'].shape} | text length: {s['input_ids'].shape[0]}")


Sample 0 | image shape: torch.Size([25988, 1176]) | text length: 6562
Sample 1 | image shape: torch.Size([25988, 1176]) | text length: 6550
Sample 2 | image shape: torch.Size([21316, 1176]) | text length: 5377
Sample 3 | image shape: torch.Size([21316, 1176]) | text length: 5381
Sample 4 | image shape: torch.Size([21316, 1176]) | text length: 5415


In [33]:
print(len(train_loader))
print(len(test_loader))

1325
386


In [34]:
sample = train_dataset[1]

print(sample.keys())        # see what the processor returned
for k, v in sample.items():
    print(k, v.shape)

dict_keys(['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw', 'labels'])
input_ids torch.Size([6550])
attention_mask torch.Size([6550])
pixel_values torch.Size([25988, 1176])
image_grid_thw torch.Size([3])
labels torch.Size([6550])


In [17]:
!pip install -q peft accelerate bitsandbytes


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForImageTextToText

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quant_config
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # common for transformers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

trainable params: 2,523,136 || all params: 8,293,898,752 || trainable%: 0.0304


In [35]:
import torch
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(model.parameters(), lr=2e-4)

model.train()
for epoch in range(2):  # start small
    loop = tqdm(train_loader)
    for batch in loop:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        loop.set_description(f"Epoch {epoch}")
        loop.set_postfix(loss=loss.item())


  0%|          | 0/1325 [00:02<?, ?it/s]


TypeError: 'NoneType' object is not iterable